# MA3632 — Workshop 5: The Learning Framework and $k$-Nearest Neighbours

This workshop accompanies the Week 5 lecture. Parts A–C build the empirical risk
minimisation framework and explore $k$-NN for classification and regression. Part D
implements cross-validation from scratch. Part E examines the effect of feature
scaling and choice of distance metric.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_digits, load_wine
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(0)

---
## Part A — Loss functions and empirical risk

The lecture defines the empirical risk of a hypothesis $h$ on a sample
$\{(x_i, y_i)\}_{i=1}^n$ as
$$
\hat{R}(h) = \frac{1}{n}\sum_{i=1}^n \ell(h(x_i), y_i).
$$
We begin by implementing the three most common loss functions by hand.

In [ ]:
def loss_01(y_hat, y):
    return (y_hat != y).astype(float)

def loss_squared(y_hat, y):
    return (y_hat - y) ** 2

def loss_absolute(y_hat, y):
    return np.abs(y_hat - y)

def empirical_risk(h_vals, y, loss_fn):
    return loss_fn(h_vals, y).mean()

# Demonstrate on a small synthetic regression example
y_demo = np.array([1.0, 2.5, 3.0, 4.2, 5.1])
c_vals = np.linspace(0, 6, 300)

risk_sq  = [empirical_risk(np.full_like(y_demo, c), y_demo, loss_squared)  for c in c_vals]
risk_abs = [empirical_risk(np.full_like(y_demo, c), y_demo, loss_absolute) for c in c_vals]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, risk, label, colour in zip(
        axes,
        [risk_sq, risk_abs],
        ["Squared loss", "Absolute loss"],
        ["steelblue", "darkorange"]):
    ax.plot(c_vals, risk, color=colour)
    ax.axvline(np.mean(y_demo),  color="steelblue",  linestyle="--", label=f"mean  = {np.mean(y_demo):.2f}")
    ax.axvline(np.median(y_demo), color="darkorange", linestyle=":",  label=f"median = {np.median(y_demo):.2f}")
    ax.set_xlabel("Constant predictor $c$")
    ax.set_ylabel("Empirical risk")
    ax.set_title(label)
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Mean of y:   {np.mean(y_demo):.4f}  (minimises squared loss)")
print(f"Median of y: {np.median(y_demo):.4f}  (minimises absolute loss)")

**In-class exercise.** The plot above confirms the standard result: the mean minimises
squared loss and the median minimises absolute loss over the class of constant predictors.
Verify analytically that the squared-loss risk $\frac{1}{n}\sum_i (c - y_i)^2$ is
minimised by $c = \bar{y}$ by differentiating with respect to $c$ and setting the
derivative to zero.

---
## Part B — $k$-NN classification

We use the Iris dataset (4 features, 3 classes, 150 samples). For visualisation we
work with the first two features only; the full four-feature model is used for the
error curves.

In [ ]:
iris = load_iris(as_frame=True)
X_ir, y_ir = iris.data.values, iris.target.values

# Split first, then fit the scaler on the training portion only
X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
    X_ir, y_ir, test_size=0.3, random_state=0, stratify=y_ir
)

sc_ir = StandardScaler()
X_tr = sc_ir.fit_transform(X_tr_raw)
X_te = sc_ir.transform(X_te_raw)

# Decision boundaries (2D) for four values of k
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
h = 0.02
# Plot bounds only (no fitting) drawn from train+test together, purely for a
# complete visual — this does not feed into the scaler or the classifier.
X_both = np.vstack([X_tr, X_te])
x1_min, x1_max = X_both[:, 0].min() - 0.5, X_both[:, 0].max() + 0.5
x2_min, x2_max = X_both[:, 1].min() - 0.5, X_both[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x1_min, x1_max, h), np.arange(x2_min, x2_max, h))
colours = ["#a8d8ea", "#f9c784", "#c8e6c9"]
cmap_bg = plt.matplotlib.colors.ListedColormap(colours)

for ax, k in zip(axes, [1, 5, 15, 50]):
    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(X_tr[:, :2], y_tr)
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, cmap=cmap_bg, alpha=0.7)
    ax.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap="Set1", s=18, edgecolors="k", linewidths=0.4)
    ax.set_title(f"$k = {k}$", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Decision boundaries (features 1 and 2 only)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Training and test error curves over k (all 4 features)
k_vals = list(range(1, 51))
train_err, test_err = [], []
for k in k_vals:
    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(X_tr, y_tr)
    train_err.append(1 - accuracy_score(y_tr, clf.predict(X_tr)))
    test_err.append(1 - accuracy_score(y_te, clf.predict(X_te)))

plt.figure(figsize=(8, 4))
plt.plot(k_vals, train_err, label="Training error", color="steelblue")
plt.plot(k_vals, test_err,  label="Test error",     color="darkorange")
plt.xlabel("$k$")
plt.ylabel("Misclassification rate")
plt.title("$k$-NN error curves — Iris (4 features, standardised)")
plt.legend()
plt.tight_layout()
plt.show()

**In-class exercise.** The decision boundaries above become smoother as $k$ increases.
Explain why $k = 1$ always achieves zero training error. At what point on the error
curve does the bias–variance trade-off appear to be best balanced?

---
## Part C — $k$-NN regression and the bias–variance decomposition

We generate a noisy sinusoidal target and estimate the bias–variance decomposition
empirically using bootstrap resamples.

In [ ]:
# Synthetic regression data
n = 150
X_reg = rng.uniform(0, 2 * np.pi, n).reshape(-1, 1)
y_reg = np.sin(X_reg.ravel()) + rng.normal(0, 0.3, n)
X_reg_tr, X_reg_te, y_reg_tr, y_reg_te = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=0
)

# Fitted curves for k = 1, 5, 20
x_plot = np.linspace(0, 2 * np.pi, 300).reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, k in zip(axes, [1, 5, 20]):
    reg = KNeighborsRegressor(n_neighbors=k)
    reg.fit(X_reg_tr, y_reg_tr)
    ax.scatter(X_reg_tr, y_reg_tr, s=14, alpha=0.5, color="steelblue")
    ax.plot(x_plot, np.sin(x_plot), "k--", linewidth=1, label="True $f$")
    ax.plot(x_plot, reg.predict(x_plot), color="darkorange", label=f"$k={k}$")
    ax.set_title(f"$k = {k}$"); ax.legend(fontsize=9)
plt.suptitle("$k$-NN regression — sinusoidal target", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Empirical bias-variance at three fixed query points
query_pts = np.array([[np.pi / 4], [np.pi], [3 * np.pi / 2]])
B = 200

for k in [1, 5, 20]:
    preds = np.zeros((B, len(query_pts)))
    for b in range(B):
        idx = rng.integers(0, len(X_reg_tr), len(X_reg_tr))
        reg = KNeighborsRegressor(n_neighbors=k)
        reg.fit(X_reg_tr[idx], y_reg_tr[idx])
        preds[b] = reg.predict(query_pts)

    true_vals = np.sin(query_pts.ravel())
    bias2 = (preds.mean(axis=0) - true_vals) ** 2
    var   = preds.var(axis=0)
    print(f"k = {k:2d}   bias^2 = {bias2.mean():.4f}   var = {var.mean():.4f}   "
          f"bias^2+var = {(bias2+var).mean():.4f}")

---
## Part D — Cross-validation and the one-standard-error rule

We implement 5-fold CV by hand on the Iris data so the mechanics are visible,
then apply the one-standard-error rule to select $k$.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=1)
k_vals = list(range(1, 31, 2))

cv_errors = np.zeros((len(k_vals), 5))
for j, k in enumerate(k_vals):
    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_tr)):
        clf = KNeighborsClassifier(n_neighbors=k)
        clf.fit(X_tr[tr_idx], y_tr[tr_idx])
        cv_errors[j, fold] = 1 - accuracy_score(y_tr[va_idx], clf.predict(X_tr[va_idx]))

mean_cv = cv_errors.mean(axis=1)
se_cv   = cv_errors.std(axis=1) / np.sqrt(5)

best_idx   = mean_cv.argmin()
threshold  = mean_cv[best_idx] + se_cv[best_idx]
ose_idx    = np.where(mean_cv <= threshold)[0][-1]   # largest k still within 1 SE

plt.figure(figsize=(9, 4))
plt.plot(k_vals, mean_cv, "o-", color="steelblue", label="CV error")
plt.fill_between(k_vals,
                 mean_cv - se_cv,
                 mean_cv + se_cv, alpha=0.2, color="steelblue")
plt.axhline(threshold,  color="darkorange", linestyle="--", label="1-SE threshold")
plt.axvline(k_vals[ose_idx], color="green", linestyle=":", label=f"1-SE choice: $k={k_vals[ose_idx]}$")
plt.xlabel("$k$"); plt.ylabel("CV misclassification rate")
plt.title("5-fold CV with one-standard-error rule — Iris")
plt.legend(); plt.tight_layout(); plt.show()

print(f"CV minimiser:        k = {k_vals[best_idx]:2d}  (error = {mean_cv[best_idx]:.4f})")
print(f"1-SE selected k:     k = {k_vals[ose_idx]:2d}  (error = {mean_cv[ose_idx]:.4f})")

In [ ]:
# Confirm against GridSearchCV
param_grid = {"n_neighbors": k_vals}
gs = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring="accuracy")
gs.fit(X_tr, y_tr)
k_star = gs.best_params_["n_neighbors"]

clf_final = KNeighborsClassifier(n_neighbors=k_vals[ose_idx])
clf_final.fit(X_tr, y_tr)
print(f"GridSearchCV exact minimiser: k = {k_star}")
print(f"Test error (1-SE model):      {1 - accuracy_score(y_te, clf_final.predict(X_te)):.4f}")
print("The manual implementation above and GridSearchCV should agree on the minimiser.")

**In-class exercise.** The 1-SE rule selects a larger $k$ than the exact CV minimiser.
Explain in one sentence why this is generally desirable from a bias–variance perspective.

---
## Part E — Distance metrics and feature scaling

This section demonstrates concretely why feature standardisation matters for $k$-NN,
and compares the Euclidean, Manhattan, and Chebyshev metrics.

We use the Digits dataset (64 pixel-intensity features, 10 classes, 1797 samples).
The raw pixel values span a wide range (0–16), but the *relative* ranges across
features vary substantially because background pixels are almost always zero while
central pixels can reach the maximum. This makes it a good test case for scaling.

### E1. The effect of standardisation

In [ ]:
digits = load_digits()
X_dg, y_dg = digits.data, digits.target

X_tr_dg, X_te_dg, y_tr_dg, y_te_dg = train_test_split(
    X_dg, y_dg, test_size=0.3, random_state=0, stratify=y_dg
)

# Without standardisation
clf_raw = KNeighborsClassifier(n_neighbors=5)
clf_raw.fit(X_tr_dg, y_tr_dg)
acc_raw = accuracy_score(y_te_dg, clf_raw.predict(X_te_dg))

# With standardisation
sc_dg = StandardScaler()
X_tr_dg_sc = sc_dg.fit_transform(X_tr_dg)
X_te_dg_sc = sc_dg.transform(X_te_dg)

clf_sc = KNeighborsClassifier(n_neighbors=5)
clf_sc.fit(X_tr_dg_sc, y_tr_dg)
acc_sc = accuracy_score(y_te_dg, clf_sc.predict(X_te_dg_sc))

print(f"Test accuracy without standardisation: {acc_raw:.4f}")
print(f"Test accuracy with standardisation:    {acc_sc:.4f}")
print()

# Show the feature-range distribution
ranges = X_dg.max(axis=0) - X_dg.min(axis=0)
print(f"Feature range statistics across 64 pixels:")
print(f"  min range = {ranges.min():.2f}")
print(f"  max range = {ranges.max():.2f}")
print(f"  mean range = {ranges.mean():.2f}")
print(f"  std of ranges = {ranges.std():.2f}")
print()
print("Corner and edge pixels (almost always zero) have near-zero range.")
print("Central pixels have large range — standardisation corrects for this.")

### E2. Euclidean, Manhattan, and Chebyshev on standardised data

In [ ]:
for metric in ["euclidean", "manhattan", "chebyshev"]:
    clf_m = KNeighborsClassifier(n_neighbors=5, metric=metric)
    clf_m.fit(X_tr_dg_sc, y_tr_dg)
    acc = accuracy_score(y_te_dg, clf_m.predict(X_te_dg_sc))
    print(f"metric = {metric:<12}  test accuracy = {acc:.4f}")

The differences between metrics are typically small on standardised data because
standardisation removes the scale differences that would otherwise make certain
features dominate distance calculations. The right metric depends on the geometry
of the problem; Euclidean is the standard default in the absence of domain knowledge.

---
## Take-home exercises

**Exercise 1.** The Wine dataset (`sklearn.datasets.load_wine`) has 13 features and
3 classes. Apply the full pipeline from Parts B and D: standardise the features, split
into training and test sets, run 5-fold cross-validation over $k \in \{1, 3, \ldots, 29\}$,
apply the one-standard-error rule to select $k$, and report the test error of the final
model. Compare your selected $k$ to the exact cross-validation minimiser.

**Exercise 2.** Weighted $k$-NN assigns each neighbour a weight proportional to the
inverse of its distance to the query point rather than equal weight. Implement this
by hand using `KNeighborsClassifier(weights='distance')` and compare the
cross-validated error curve to the uniform-weight version on the Iris data. Does
inverse-distance weighting improve performance for small $k$?

**Exercise 3.** Repeat the bootstrap bias–variance experiment from Part C, but now
for $k$-NN classification on the Digits dataset. Fix three values of $k$
(e.g.\ $k = 1, 7, 25$) and use $B = 100$ bootstrap resamples. At ten randomly
chosen test points, estimate the squared bias and variance of the predicted class
probability for class 0. Report how bias and variance change as $k$ increases.

**Exercise 4.** Compare LOOCV to 5-fold CV on the Digits dataset over
$k \in \{1, 3, 5, 7, 9\}$. For which values of $k$ do the two estimators disagree
most? What does this suggest about the variance of each estimator?